# Lesson 4 — Fine-tuning a Summarization LLM

This notebook fine-tunes an open-source language model for document
summarization using supervised fine-tuning and QLoRA.

Pipeline:

1. Verify the GPU environment.
2. Install the training libraries.
3. Load and inspect the dataset.
4. Load the quantized base model.
5. Attach LoRA adapters.
6. Format the training prompts.
7. Train and evaluate the model.
8. Save the adapter to Hugging Face.

In [3]:
import platform
import sys

import torch


print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No NVIDIA GPU detected. Select Runtime → Change runtime type → GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_properties = torch.cuda.get_device_properties(0)
gpu_memory_gb = gpu_properties.total_memory / (1024**3)

print("GPU:", gpu_name)
print(f"GPU memory: {gpu_memory_gb:.2f} GB")
print("CUDA version:", torch.version.cuda)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB
CUDA version: 12.8


In [4]:
!nvidia-smi

Mon Aug  3 19:18:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
%pip install --quiet --upgrade unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 

In [6]:
import unsloth

import datasets
import transformers
import trl


print("Unsloth:", unsloth.__version__)
print("Datasets:", datasets.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: 2026.8.2
Datasets: 4.3.0
Transformers: 5.5.0
TRL: 0.24.0


## 3. Load and inspect the summarization dataset

The dataset contains document-summary pairs generated during the dataset
generation pipeline.

- `instruction`: the source document to summarize.
- `answer`: the expected high-quality summary.
- `train`: teaches the model.
- `validation`: monitors training and helps tune configuration.
- `test`: evaluates the final model on unseen samples.

In [7]:
from datasets import load_dataset


DATASET_ID = (
    "pauliusztin/"
    "second_brain_course_summarization_task"
)

dataset = load_dataset(DATASET_ID)

print(dataset)

for split_name, split_dataset in dataset.items():
    print(
        f"{split_name}: "
        f"{len(split_dataset)} samples, "
        f"columns={split_dataset.column_names}"
    )

README.md:   0%|          | 0.00/529 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.25MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.11MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/575 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/72 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/72 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'answer'],
        num_rows: 575
    })
    validation: Dataset({
        features: ['instruction', 'answer'],
        num_rows: 72
    })
    test: Dataset({
        features: ['instruction', 'answer'],
        num_rows: 72
    })
})
train: 575 samples, columns=['instruction', 'answer']
validation: 72 samples, columns=['instruction', 'answer']
test: 72 samples, columns=['instruction', 'answer']


In [8]:
sample = dataset["train"][0]

print("Instruction characters:", len(sample["instruction"]))
print("Answer characters:", len(sample["answer"]))

print("\nInstruction preview:")
print(sample["instruction"][:500])

print("\nExpected answer:")
print(sample["answer"])

Instruction characters: 383
Answer characters: 285

Instruction preview:
# Notes



<child_page>
# Design Patterns

# Training code

The most natural way of splitting the training code:
- Dataset
- DatasetLoader
- Model
- ModelFactory
- Trainer (takes in the dataset and model)
- Evaluator

# Serving code

[Infrastructure]Model (takes in the trained model)
	- register
	- deploy
</child_page>


---

# Resources [Community]

# Resources [Science]

# Tools

Expected answer:
```markdown
# TL;DR Summary

## Design Patterns
- **Training Code Structure**: Key components include Dataset, DatasetLoader, Model, ModelFactory, Trainer, and Evaluator.
- **Serving Code Infrastructure**: Involves Model registration and deployment.

## Tags
- Generative AI
- LLMs
```


In [9]:
required_columns = {"instruction", "answer"}

for split_name, split_dataset in dataset.items():
    missing_columns = (
        required_columns
        - set(split_dataset.column_names)
    )

    if missing_columns:
        raise ValueError(
            f"{split_name} is missing columns: "
            f"{missing_columns}"
        )

print("Dataset schema is valid.")

Dataset schema is valid.


## 4. Format samples using an Alpaca prompt

The trainer needs one complete text sequence containing:

1. The task the model must perform.
2. The source document.
3. The expected response.

During training, the model learns that the response should follow the
instruction and input sections.

In [10]:
ALPACA_PROMPT = """Below is an instruction that describes a task,
paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
You are a helpful assistant specialized in summarizing documents.
Generate a concise TL;DR summary in Markdown format with a maximum
of 512 characters. Highlight the most significant insights.

### Input:
{document}

### Response:
{summary}"""

In [11]:
def format_training_example(
    document: str,
    summary: str,
) -> str:
    return ALPACA_PROMPT.format(
        document=document.strip(),
        summary=summary.strip(),
    )

In [12]:
formatted_sample = format_training_example(
    document=dataset["train"][0]["instruction"],
    summary=dataset["train"][0]["answer"],
)

print(formatted_sample[:1500])

Below is an instruction that describes a task,
paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
You are a helpful assistant specialized in summarizing documents.
Generate a concise TL;DR summary in Markdown format with a maximum
of 512 characters. Highlight the most significant insights.

### Input:
# Notes



<child_page>
# Design Patterns

# Training code

The most natural way of splitting the training code:
- Dataset
- DatasetLoader
- Model
- ModelFactory
- Trainer (takes in the dataset and model)
- Evaluator

# Serving code

[Infrastructure]Model (takes in the trained model)
	- register
	- deploy
</child_page>


---

# Resources [Community]

# Resources [Science]

# Tools

### Response:
```markdown
# TL;DR Summary

## Design Patterns
- **Training Code Structure**: Key components include Dataset, DatasetLoader, Model, ModelFactory, Trainer, and Evaluator.
- **Serving Code Infrastructure**: Involves Model

## 5. Load the four-bit base model

We load Llama 3.1 8B Instruct using four-bit quantization.

- The base model provides its existing language abilities.
- Four-bit quantization reduces its VRAM usage.
- The tokenizer converts text into token IDs and converts generated IDs
  back into text.
- No LoRA adapters have been attached yet.

In [13]:
import torch

from unsloth import FastLanguageModel


MODEL_ID = (
    "unsloth/"
    "Meta-Llama-3.1-8B-Instruct-unsloth-bnb-4bit"
)

MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT = True

torch.cuda.empty_cache()

memory_before_gb = (
    torch.cuda.memory_allocated() / (1024**3)
)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)

memory_after_gb = (
    torch.cuda.memory_allocated() / (1024**3)
)

print("Model:", MODEL_ID)
print("Model class:", type(model).__name__)
print("Tokenizer class:", type(tokenizer).__name__)
print("Maximum sequence length:", MAX_SEQ_LENGTH)
print("Loaded in four-bit:", LOAD_IN_4BIT)
print(f"GPU memory before: {memory_before_gb:.2f} GB")
print(f"GPU memory after: {memory_after_gb:.2f} GB")
print(
    f"Model memory increase: "
    f"{memory_after_gb - memory_before_gb:.2f} GB"
)

==((====))==  Unsloth 2026.8.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-unsloth-bnb-4bit as a legacy tokenizer.


Model: unsloth/Meta-Llama-3.1-8B-Instruct-unsloth-bnb-4bit
Model class: LlamaForCausalLM
Tokenizer class: TokenizersBackend
Maximum sequence length: 4096
Loaded in four-bit: True
GPU memory before: 0.01 GB
GPU memory after: 5.60 GB
Model memory increase: 5.59 GB


In [14]:
def count_parameters(model) -> tuple[int, int]:
    total_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    trainable_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    return total_parameters, trainable_parameters


total_before, trainable_before = count_parameters(model)

print(f"Total parameters before LoRA: {total_before:,}")
print(f"Trainable before LoRA: {trainable_before:,}")

Total parameters before LoRA: 4,628,680,704
Trainable before LoRA: 701,763,584


In [15]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

Unsloth 2026.8.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [16]:
total_after, trainable_after = count_parameters(model)

trainable_percentage = (
    100 * trainable_after / total_after
)

print(f"Total parameters after LoRA: {total_after:,}")
print(f"Trainable after LoRA: {trainable_after:,}")
print(f"Trainable percentage: {trainable_percentage:.4f}%")

Total parameters after LoRA: 4,670,623,744
Trainable after LoRA: 41,943,040
Trainable percentage: 0.8980%


In [17]:
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [18]:
trainable_parameters, total_parameters = (
    model.get_nb_trainable_parameters()
)

print(f"Logical total: {total_parameters:,}")
print(f"Trainable: {trainable_parameters:,}")
print(
    "Trainable percentage:",
    f"{100 * trainable_parameters / total_parameters:.4f}%",
)

Logical total: 8,072,204,288
Trainable: 41,943,040
Trainable percentage: 0.5196%


## 7. Prepare the complete training dataset

Each dataset row is converted into one Alpaca-formatted text sequence.

An end-of-sequence token is appended so the model learns where the expected
response finishes.

In [19]:
EOS_TOKEN = tokenizer.eos_token

print("EOS token:", repr(EOS_TOKEN))
print("EOS token ID:", tokenizer.eos_token_id)

EOS token: '<|eot_id|>'
EOS token ID: 128009


In [20]:
def format_training_batch(
    examples: dict[str, list[str]],
) -> dict[str, list[str]]:
    documents = examples["instruction"]
    summaries = examples["answer"]

    formatted_texts = [
        format_training_example(
            document=document,
            summary=summary,
        )
        + EOS_TOKEN
        for document, summary in zip(
            documents,
            summaries,
        )
    ]

    return {"text": formatted_texts}

In [21]:
formatted_dataset = dataset.map(
    format_training_batch,
    batched=True,
    desc="Formatting training examples",
)

print(formatted_dataset)
print(
    "Training columns:",
    formatted_dataset["train"].column_names,
)

Formatting training examples:   0%|          | 0/575 [00:00<?, ? examples/s]

Formatting training examples:   0%|          | 0/72 [00:00<?, ? examples/s]

Formatting training examples:   0%|          | 0/72 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'answer', 'text'],
        num_rows: 575
    })
    validation: Dataset({
        features: ['instruction', 'answer', 'text'],
        num_rows: 72
    })
    test: Dataset({
        features: ['instruction', 'answer', 'text'],
        num_rows: 72
    })
})
Training columns: ['instruction', 'answer', 'text']


In [22]:
formatted_text = formatted_dataset["train"][0]["text"]

print(formatted_text[:1500])
print("\nEnds with EOS:", formatted_text.endswith(EOS_TOKEN))

Below is an instruction that describes a task,
paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
You are a helpful assistant specialized in summarizing documents.
Generate a concise TL;DR summary in Markdown format with a maximum
of 512 characters. Highlight the most significant insights.

### Input:
# Notes



<child_page>
# Design Patterns

# Training code

The most natural way of splitting the training code:
- Dataset
- DatasetLoader
- Model
- ModelFactory
- Trainer (takes in the dataset and model)
- Evaluator

# Serving code

[Infrastructure]Model (takes in the trained model)
	- register
	- deploy
</child_page>


---

# Resources [Community]

# Resources [Science]

# Tools

### Response:
```markdown
# TL;DR Summary

## Design Patterns
- **Training Code Structure**: Key components include Dataset, DatasetLoader, Model, ModelFactory, Trainer, and Evaluator.
- **Serving Code Infrastructure**: Involves Model

In [23]:
token_ids = tokenizer(
    formatted_text,
    add_special_tokens=False,
)["input_ids"]

print("Sample token count:", len(token_ids))
print("Maximum allowed:", MAX_SEQ_LENGTH)
print(
    "Will be truncated:",
    len(token_ids) > MAX_SEQ_LENGTH,
)

Sample token count: 234
Maximum allowed: 4096
Will be truncated: False


## 8. Configure supervised fine-tuning

This configuration defines:

- How much data enters the GPU at once.
- How many training steps run.
- How quickly the adapter parameters change.
- Which numerical precision and optimizer are used.
- Where the resulting checkpoints are stored.

This is a short educational training run on a T4 GPU.

In [24]:
from trl import SFTConfig, SFTTrainer


if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


training_config = SFTConfig(
    output_dir="outputs",

    # Dataset processing
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,

    # Effective batch size:
    # 1 sample × 4 accumulated steps = 4
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    # Short educational run
    warmup_steps=5,
    max_steps=25,

    # LoRA commonly uses a higher learning rate
    learning_rate=2e-4,

    # T4 configuration
    fp16=True,
    bf16=False,

    # Optimizer and scheduling
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",

    # Logging and reproducibility
    logging_steps=1,
    seed=3407,
    report_to="none",

    # Avoid checkpoint overhead during the first run
    save_strategy="no",
    eval_strategy="no",
)

In [25]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=formatted_dataset["train"],
    args=training_config,
)

print("Trainer created successfully.")
print("Training samples:", len(formatted_dataset["train"]))
print("Maximum steps:", training_config.max_steps)
print(
    "Effective batch size:",
    (
        training_config.per_device_train_batch_size
        * training_config.gradient_accumulation_steps
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/575 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Trainer created successfully.
Training samples: 575
Maximum steps: 25
Effective batch size: 4


In [26]:
test_split = formatted_dataset["test"]

test_index = min(
    range(len(test_split)),
    key=lambda index: len(
        test_split[index]["instruction"]
    ),
)

test_document = test_split[test_index]["instruction"]
expected_summary = test_split[test_index]["answer"]

print("Selected test index:", test_index)
print("Document characters:", len(test_document))
print("Expected summary characters:", len(expected_summary))

Selected test index: 0
Document characters: 383
Expected summary characters: 280


In [27]:
def format_inference_prompt(document: str) -> str:
    return ALPACA_PROMPT.format(
        document=document.strip(),
        summary="",
    )


inference_prompt = format_inference_prompt(
    test_document
)

print(inference_prompt[:1000])

Below is an instruction that describes a task,
paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
You are a helpful assistant specialized in summarizing documents.
Generate a concise TL;DR summary in Markdown format with a maximum
of 512 characters. Highlight the most significant insights.

### Input:
# Notes



<child_page>
# Design Patterns

# Training code

The most natural way of splitting the training code:
- Dataset
- DatasetLoader
- Model
- ModelFactory
- Trainer (takes in the dataset and model)
- Evaluator

# Serving code

[Infrastructure]Model (takes in the trained model)
	- register
	- deploy
</child_page>


---

# Resources [Community]

# Resources [Science]

# Tools

### Response:



In [28]:
model.eval()

inputs = tokenizer(
    inference_prompt,
    return_tensors="pt",
    add_special_tokens=True,
).to("cuda")

input_token_count = inputs["input_ids"].shape[1]

if input_token_count >= MAX_SEQ_LENGTH:
    raise ValueError(
        f"Test prompt has {input_token_count} tokens, "
        f"but the maximum is {MAX_SEQ_LENGTH}"
    )

with torch.no_grad():
    generated_output = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        use_cache=True,
    )

generated_tokens = generated_output[
    0,
    input_token_count:,
]

baseline_summary = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True,
).strip()

print("Input tokens:", input_token_count)

print("\nBaseline summary:")
print(baseline_summary)

print("\nExpected summary:")
print(expected_summary)

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input tokens: 167

Baseline summary:
Here is a concise TL;DR summary in Markdown format:

### Design Patterns for Training and Serving Code
#### Key Insights

*   **Modularize Training Code**: Split into distinct components:
    *   Dataset
    *   DatasetLoader
    *   Model
    *   ModelFactory
    *   Trainer (takes in dataset and model)
    *   Evaluator
*   **Simplify Serving Code**: Use a Model component that takes in the trained model:
    *   register
    *   deploy
*   **Separate Concerns**: Keep training and serving code organized and easy to maintain.

This summary

Expected summary:
```markdown
# TL;DR Summary

## Design Patterns
- **Training Code Structure**: Key components include Dataset, DatasetLoader, Model, ModelFactory, Trainer, and Evaluator.
- **Serving Code**: Infrastructure for Model registration and deployment.

## Tags
- Generative AI
- LLMs
```


In [29]:
model.train()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

## 10. Run supervised fine-tuning

This short run performs 25 optimizer updates on the LoRA adapters.

The quantized base model remains frozen. We monitor:

- Training loss
- Runtime
- GPU peak memory
- Samples processed per second

In [30]:
import time

import torch


torch.cuda.reset_peak_memory_stats()

memory_before_training_gb = (
    torch.cuda.memory_reserved() / (1024**3)
)

training_start_time = time.perf_counter()

training_result = trainer.train()

training_duration_seconds = (
    time.perf_counter() - training_start_time
)

peak_memory_gb = (
    torch.cuda.max_memory_reserved() / (1024**3)
)

additional_peak_memory_gb = (
    peak_memory_gb - memory_before_training_gb
)

print("\nTraining completed.")
print(
    f"Duration: "
    f"{training_duration_seconds / 60:.2f} minutes"
)
print(
    f"Memory before training: "
    f"{memory_before_training_gb:.2f} GB"
)
print(
    f"Peak memory: "
    f"{peak_memory_gb:.2f} GB"
)
print(
    f"Additional training memory: "
    f"{additional_peak_memory_gb:.2f} GB"
)

print("\nTraining metrics:")

for metric_name, metric_value in training_result.metrics.items():
    print(f"{metric_name}: {metric_value}")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 575 | Num Epochs = 1 | Total steps = 25
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.465401
2,1.559251
3,1.443994
4,1.428834
5,1.402659
6,1.750359
7,1.410465
8,1.562114
9,1.508782
10,1.299045



Training completed.
Duration: 21.13 minutes
Memory before training: 6.21 GB
Peak memory: 9.80 GB
Additional training memory: 3.59 GB

Training metrics:
train_runtime: 1263.8297
train_samples_per_second: 0.079
train_steps_per_second: 0.02
total_flos: 1.6268450664579072e+16
train_loss: 1.3638545966148377
epoch: 0.17391304347826086


In [31]:
loss_history = [
    entry["loss"]
    for entry in trainer.state.log_history
    if "loss" in entry
]

print("Logged loss values:", len(loss_history))

if loss_history:
    print("First loss:", loss_history[0])
    print("Final loss:", loss_history[-1])
    print("Lowest loss:", min(loss_history))

Logged loss values: 25
First loss: 1.46540105342865
Final loss: 1.5861468315124512
Lowest loss: 0.9374918341636658


In [32]:
FastLanguageModel.for_inference(model)

model.eval()

inputs = tokenizer(
    inference_prompt,
    return_tensors="pt",
    add_special_tokens=True,
).to("cuda")

input_token_count = inputs["input_ids"].shape[1]

with torch.no_grad():
    fine_tuned_output = model.generate(
        **inputs,
        max_length=input_token_count + 128,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )

fine_tuned_tokens = fine_tuned_output[
    0,
    input_token_count:,
]

fine_tuned_summary = tokenizer.decode(
    fine_tuned_tokens,
    skip_special_tokens=True,
).strip()

In [33]:
print("=== BASELINE ===")
print(baseline_summary)

print("\n=== FINE-TUNED ===")
print(fine_tuned_summary)

print("\n=== EXPECTED ===")
print(expected_summary)

print("\nCharacter counts:")
print("Baseline:", len(baseline_summary))
print("Fine-tuned:", len(fine_tuned_summary))
print("Expected:", len(expected_summary))

=== BASELINE ===
Here is a concise TL;DR summary in Markdown format:

### Design Patterns for Training and Serving Code
#### Key Insights

*   **Modularize Training Code**: Split into distinct components:
    *   Dataset
    *   DatasetLoader
    *   Model
    *   ModelFactory
    *   Trainer (takes in dataset and model)
    *   Evaluator
*   **Simplify Serving Code**: Use a Model component that takes in the trained model:
    *   register
    *   deploy
*   **Separate Concerns**: Keep training and serving code organized and easy to maintain.

This summary

=== FINE-TUNED ===
```markdown
# TL;DR

## Summary

This document outlines a structured approach to organizing training and serving code for machine learning models. It proposes a modular architecture with clear separation of concerns, including:

- **Training code**: Dataset, DatasetLoader, Model, ModelFactory, Trainer, Evaluator
- **Serving code**: Infrastructure, Model (register, deploy)

This structure facilitates maintainabilit

In [34]:
from pathlib import Path
import json


ADAPTER_DIRECTORY = Path(
    "/content/second-brain-summarizer-lora-25steps"
)

ADAPTER_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

model.save_pretrained(
    ADAPTER_DIRECTORY.as_posix(),
    safe_serialization=True,
)

tokenizer.save_pretrained(
    ADAPTER_DIRECTORY.as_posix(),
)

Unsloth: Restored added_tokens_decoder metadata in /content/second-brain-summarizer-lora-25steps/tokenizer_config.json.


('/content/second-brain-summarizer-lora-25steps/tokenizer_config.json',
 '/content/second-brain-summarizer-lora-25steps/chat_template.jinja',
 '/content/second-brain-summarizer-lora-25steps/tokenizer.json')

In [35]:
experiment_metadata = {
    "base_model": MODEL_ID,
    "dataset": DATASET_ID,
    "max_sequence_length": MAX_SEQ_LENGTH,
    "load_in_4bit": LOAD_IN_4BIT,
    "lora_rank": 16,
    "lora_alpha": 16,
    "max_steps": training_config.max_steps,
    "effective_batch_size": 4,
    "trainable_parameters": 41_943_040,
    "trainable_percentage": 0.5196,
    "training_metrics": training_result.metrics,
    "evaluation": {
        "baseline_characters": len(baseline_summary),
        "fine_tuned_characters": len(fine_tuned_summary),
        "expected_characters": len(expected_summary),
        "baseline_summary": baseline_summary,
        "fine_tuned_summary": fine_tuned_summary,
        "expected_summary": expected_summary,
    },
}

metadata_path = (
    ADAPTER_DIRECTORY
    / "experiment_metadata.json"
)

metadata_path.write_text(
    json.dumps(
        experiment_metadata,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

2341

In [36]:
for file_path in sorted(
    ADAPTER_DIRECTORY.iterdir()
):
    file_size_mb = (
        file_path.stat().st_size
        / (1024**2)
    )

    print(
        file_path.name,
        f"{file_size_mb:.2f} MB",
    )

README.md 0.01 MB
adapter_config.json 0.00 MB
adapter_model.safetensors 160.06 MB
chat_template.jinja 0.00 MB
experiment_metadata.json 0.00 MB
tokenizer.json 16.41 MB
tokenizer_config.json 0.05 MB


In [37]:
from google.colab import userdata
from huggingface_hub import HfApi, login

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise RuntimeError(
        "HF_TOKEN is missing. Add it through Colab Secrets."
    )

login(
    token=hf_token,
    add_to_git_credential=False,
)

api = HfApi(token=hf_token)
account = api.whoami()

HF_USERNAME = account["name"]

print("Connected Hugging Face account:", HF_USERNAME)

SecretNotFoundError: Secret HF_TOKEN does not exist.

In [ ]:
import gc
import torch

from unsloth import FastLanguageModel


# Release the training objects from GPU memory.
del trainer
del model

gc.collect()
torch.cuda.empty_cache()


# Load the saved LoRA adapter.
reloaded_model, reloaded_tokenizer = (
    FastLanguageModel.from_pretrained(
        model_name=ADAPTER_DIRECTORY.as_posix(),
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
)

FastLanguageModel.for_inference(reloaded_model)


# Use the same test document used earlier.
prompt = ALPACA_PROMPT.format(
    document=test_document.strip(),
    summary="",
)

inputs = reloaded_tokenizer(
    prompt,
    return_tensors="pt",
).to("cuda")

with torch.inference_mode():
    output_ids = reloaded_model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.0,
        use_cache=True,
    )

input_length = inputs["input_ids"].shape[1]

reloaded_summary = reloaded_tokenizer.decode(
    output_ids[0][input_length:],
    skip_special_tokens=True,
)

print("Summary generated by the reloaded adapter:\n")
print(reloaded_summary)

==((====))==  Unsloth 2026.8.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]